# β-VAE ICLR 2017: reproducing Figure 7A and 7B (from scratch)

This Colab/PyTorch notebook reproduces **Figure 7A (β = 4) and Figure 7B (β = 1)** using the attached Higgins et al. paper only for the method (§2, §4.2, Fig. 7 caption, Appendix A.1/Table 1, Appendix A.4). No existing implementation of the paper is used. Data come directly from the `.npz` hosted in the original [Google DeepMind dSprites dataset repository](https://github.com/google-deepmind/dsprites-dataset). **That repository does not publish a GitHub Release**; the official repository's checked-in `.npz` is downloaded instead. The notebook records its SHA-256.

**What is paper-specified:** 737,280 binary 64×64 dSprites images; factors shape (3), scale (6), rotation (40), x (32), y (32). Each β has an identical encoder: 4096→1200→1200 with ReLU, 10 diagonal Gaussian latents; decoder 10→1200→1200→1200→4096 with Tanh hidden activations, Bernoulli likelihood; isotropic normal prior; Adagrad learning rate 0.01. The loss per image sums the 4096 Bernoulli negative log-likelihoods and adds `β × sum_10(KL)`. All models train on **unlabeled** images. Dataset-provided `latents_classes` and `latents_values` are used only for **post-training evaluation/visualization** (and selecting the common anchor), never as supervised targets.

**What Figure 7 actually shows:** For each unit, a joint 32×32 **position X/Y heatmap**, factor responses for scale and rotation separated by object identity, and an independent one-latent decode traversal. The Figure 7 caption (paper p.10) explicitly orders columns by *learned Gaussian posterior variance* and traverses over **±3 standard deviations of the unit-normal prior**, keeping the other 9 units at values obtained by inference on **an image from the dataset**. As a substitute for the figure's unspecified image, we choose one dataset image with factor-class tuple `[color=0, shape=oval(1), scale=3, rotation=0, x=16, y=16]` for all models/seeds. Its non-traversed coordinates use the inferred **posterior means** (a deliberate inference choice). Nine equispaced values over [-3,+3] (number of frames unspecified) are decoded; each row is an interpolated latent value, not a different seed image. The five lowest-variance units and the single highest-variance ('unused example') unit are shown in the primary Figure-7-like panels. Supplemental plots display **every one of the ten units**, including marginal X and Y curves. Latents are never manually aligned to ground-truth factors; sorting is by mean posterior variance, ascending, with latent index breaking ties.

**Not stated by the paper:** training duration/number of epochs or updates, minibatch size, explicit initialization/optimizer epsilon, exact data sampler, specific Figure 7 trained replica, exact traversal seed image, exact number of traversal samples, whether to fix posterior samples or posterior means, classifier-irrelevant reconstruction evaluation, and exactly which unused unit was shown. Decisions here: 20,000 updates per replica, minibatch 256, standard PyTorch Linear initialization, uniform image sampling with replacement, default PyTorch Adagrad epsilon, three independent seeds per β, all replicas kept (no cherry-picking), common centered oval anchor, posterior means to eliminate sampling noise, 9 traversal points, and last-ranked variance latent as unused example. We report objective-compatible **Monte Carlo reconstruction NLL** on the same fixed in-sample 16,384-image probe for all runs, plus deterministic at-posterior-mean NLL and **full-dataset** per-dimension KL. Probe images are in-sample because the paper does not describe a train/validation split. A higher training budget may be needed; no exact numerical reproduction is guaranteed without the authors' unspecified settings.

**Multiple seeds:** Appendix A.4 reports *ten* independently seeded replicas for its Figure 6 disentanglement benchmark, three independent classifier evaluations of each, and dropping the bottom 50% of classifier scores. Our **three per β** is a Colab-resource compromise, not a claim to reproduce that quantitative scoring protocol. **Every run is reported**, regardless of quality. For the primary side-by-side panels, the lowest fixed seed is selected independently of observed performance; the other seeds' panels are saved too. For every replica the notebook saves a resumable model+Adagrad checkpoint, training history, numerical JSON report, and NPZ containing the *trained model's* factor means/traversal images. Plot cells read those latter artifacts **only**, with all numerical display settings written to saved run metadata; they never load the original dataset or invoke an encoder/decoder.

**Run in Colab:** Runtime → Change runtime type → **T4 GPU**. This is a sizable 6-model training job; checkpoint resume and an optional Drive storage flag support Colab reconnects. The numeric results are intentionally empty until **you execute the notebook**; no untrained or invented outcome is presented.

## 1. Configuration (all numerical plot settings live here)

In [ ]:
import os, random, json, hashlib, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

C = {
 'download_url': 'https://raw.githubusercontent.com/google-deepmind/dsprites-dataset/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz',
 'filename': 'dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz',
 'betas': [4.0, 1.0], 'seeds': [7, 19, 41],
 'input_dim': 4096, 'side': 64, 'hidden': 1200, 'latent_dim': 10,
 'lr': 0.01, 'train_steps': 20000, 'batch_size': 256,
 'save_every': 2000, 'log_every': 500,
 'eval_batch': 1024, 'analysis_batch': 1024,
 'probe_n': 16384, 'probe_seed': 1337, 'eval_epsilon_seed': 2917,
 'factor_names': ['color','shape','scale','rotation','position_x','position_y'],
 'shape_names': ['square','oval','heart'],
 'anchor_classes': [0,1,3,0,16,16],
 'traversal_values': np.linspace(-3.,3.,9).tolist(), 'top_count': 5,
 'plot': {
   'dpi': 140, 'main_size': [15,15], 'factor_size': [20,10], 'traversal_size': [20,12],
   'cmap_position': 'coolwarm', 'cmap_image': 'gray_r',
   'shape_colors': ['red','green','blue'], 'top_ratios': [1.8,1.2,1.2],
   'image_ratio': 1.0, 'hspace': 0.18, 'wspace': 0.32,
   'line_width': 1.4, 'alpha': 0.85, 'small_font': 7, 'label_font': 8,
   'title_font': 10, 'image_low': 0.0, 'image_high': 1.0,
   'scale_xticks': [0,3,5], 'rotation_xticks': [0,20,39],
 }
}
USE_GOOGLE_DRIVE = False  # Change to True to persist outputs across Colab resets.
if USE_GOOGLE_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive')
 ROOT = Path('/content/drive/MyDrive/beta_vae_figure7_from_scratch')
else:
 ROOT = Path('/content/beta_vae_figure7_from_scratch')
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT/'dataset'
DATA_DIR.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE, '| save to:', ROOT)
if DEVICE.type != 'cuda':
 raise RuntimeError('Enable a Colab T4 GPU before running the full six-run experiment.')
torch.backends.cuda.matmul.allow_tf32 = False

## 2. Official `.npz` and its ground-truth labels

In [ ]:
data_path = DATA_DIR/C['filename']
if not data_path.exists():
 req = urllib.request.Request(C['download_url'], headers={'User-Agent':'dSprites-Colab-Experiment'})
 temp = data_path.with_suffix('.partial')
 with urllib.request.urlopen(req,timeout=180) as source, open(temp,'wb') as destination:
  while True:
   piece=source.read(1024*1024)
   if not piece: break
   destination.write(piece)
 os.replace(temp,data_path)
sha=hashlib.sha256()
with open(data_path,'rb') as stream:
 for chunk in iter(lambda:stream.read(1024*1024),b''): sha.update(chunk)
DATA_SHA=sha.hexdigest()
# latin1 is needed for the archive's Python-2-era metadata object.
with np.load(data_path, allow_pickle=True, encoding='latin1') as data:
 imgs=data['imgs']
 labels=data['latents_classes'].astype(np.int16,copy=False)
 values=data['latents_values']
 metadata=data['metadata'][()]
sizes=labels.max(axis=0).astype(int)+1
assert imgs.shape==(737280,64,64) and imgs.dtype==np.uint8
assert tuple(sizes)==(1,3,6,40,32,32)
assert labels.shape==(len(imgs),len(C['factor_names']))
assert int(np.prod(sizes))==len(imgs)
assert np.all((imgs[:1000]==0)|(imgs[:1000]==1))
anchor=np.array(C['anchor_classes'],dtype=np.int16)
anchor_candidates=np.flatnonzero(np.all(labels==anchor,axis=1))
assert len(anchor_candidates)==1
anchor_id=int(anchor_candidates.item())
print('Images:',imgs.shape,'| class factors:',sizes.tolist(),'| SHA-256:',DATA_SHA)
print('Fixed traversal seed image row:',anchor_id,'| actual labels:',labels[anchor_id].tolist())

## 3. Model and loss — no reused code

In [ ]:
class BetaVAE(nn.Module):
 def __init__(self,config):
  super().__init__()
  h=config['hidden']; z=config['latent_dim']; n=config['input_dim']
  self.encoder=nn.Sequential(nn.Linear(n,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU())
  self.mu=nn.Linear(h,z)
  self.logvar=nn.Linear(h,z)
  self.decoder=nn.Sequential(nn.Linear(z,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),
                             nn.Linear(h,h),nn.Tanh(),nn.Linear(h,n))
 def encode(self,x):
  h=self.encoder(x)
  return self.mu(h),self.logvar(h)
 def decode_logits(self,z): return self.decoder(z)
 def forward(self,x):
  mu,lv=self.encode(x)
  z=mu+torch.exp(0.5*lv)*torch.randn_like(mu)
  return self.decode_logits(z),mu,lv

def loss_terms(logits,x,mu,lv,beta):
 recon=F.binary_cross_entropy_with_logits(logits,x,reduction='none').sum(dim=1).mean()
 kl_dim=(mu.square()+lv.exp()-1-lv).mul(0.5).mean(dim=0)
 return recon+beta*kl_dim.sum(),recon,kl_dim

def rng_seed(seed):
 random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def folder(beta,seed): return ROOT/f'beta{beta:g}_seed{seed}'

def write_json(value,path):
 tmp=path.with_suffix('.partial')
 with open(tmp,'w',encoding='utf-8') as f: json.dump(value,f,indent=2)
 os.replace(tmp,path)

def write_checkpoint(value,path):
 tmp=path.with_suffix('.partial')
 torch.save(value,tmp)
 os.replace(tmp,path)

# Before a long fit, verify all latent shapes and finite loss on the actual GPU.
test_model=BetaVAE(C).to(DEVICE)
with torch.no_grad():
 test_x=torch.zeros((2,C['input_dim']),device=DEVICE)
 logits,mu,lv=test_model(test_x)
 objective,rec,kl=loss_terms(logits,test_x,mu,lv,C['betas'][0])
 assert logits.shape==test_x.shape and kl.shape==(C['latent_dim'],) and torch.isfinite(objective)
del test_model,test_x,logits,mu,lv,objective,rec,kl
torch.cuda.empty_cache()
print('Model shape / Bernoulli NLL / KL smoke test passed.')

## 4. Train / resume each seed and save checkpoints

In [ ]:
def fit(beta,seed):
 d=folder(beta,seed)
 d.mkdir(parents=True,exist_ok=True)
 p=d/'checkpoint.pt'
 rng_seed(seed)
 model=BetaVAE(C).to(DEVICE)
 opt=torch.optim.Adagrad(model.parameters(),lr=C['lr'])
 sampler=torch.Generator(device='cpu').manual_seed(seed+10000)
 history=[]; beginning=0
 if p.exists():
  snap=torch.load(p,map_location='cpu',weights_only=False)
  if snap['dataset_sha']!=DATA_SHA or snap['config']!=C:
   raise ValueError('Checkpoint config/dataset differs; use a fresh output directory')
  model.load_state_dict(snap['model'])
  opt.load_state_dict(snap['optimizer'])
  sampler.set_state(snap['sampler_rng'])
  torch.set_rng_state(snap['cpu_rng'])
  if snap['gpu_rng'] is not None: torch.cuda.set_rng_state_all(snap['gpu_rng'])
  history=list(snap['history']); beginning=int(snap['step'])
  print('Resume',d.name,'at step',beginning)
 else: print('New training:',d.name)
 model.train()
 batch_rec=batch_kl=0.0; count=0
 for step in range(beginning+1,C['train_steps']+1):
  ix=torch.randint(len(imgs),(C['batch_size'],),generator=sampler).numpy()
  x=torch.from_numpy(imgs[ix]).to(device=DEVICE,dtype=torch.float32).flatten(start_dim=1)
  opt.zero_grad(set_to_none=True)
  logits,mu,lv=model(x)
  obj,rec,kl=loss_terms(logits,x,mu,lv,beta)
  obj.backward(); opt.step()
  batch_rec+=float(rec.detach()); batch_kl+=float(kl.detach().sum()); count+=1
  if step%C['log_every']==0 or step==C['train_steps']:
   entry={'step':step,'training_recon_bce':batch_rec/count,'training_kl':batch_kl/count}
   history.append(entry)
   print(f'  step {step:>6}: sampled BCE={entry["training_recon_bce"]:.2f}, ΣKL={entry["training_kl"]:.2f}')
   batch_rec=batch_kl=0.0; count=0
  if step%C['save_every']==0 or step==C['train_steps']:
   state={'model':model.state_dict(),'optimizer':opt.state_dict(),
          'sampler_rng':sampler.get_state(),'cpu_rng':torch.get_rng_state(),
          'gpu_rng':torch.cuda.get_rng_state_all(),'step':step,
          'beta':beta,'seed':seed,'config':C,'dataset_sha':DATA_SHA,'history':history}
   write_checkpoint(state,p)
   write_json({'beta':beta,'seed':seed,'step':step,'history':history},d/'training_history.json')
 del model,opt
 torch.cuda.empty_cache()

## 5. Checkpoint-derived reconstruction NLL, factor responses, full-dataset KL and traversals

In [ ]:
@torch.inference_mode()
def probe_recon(model):
 ids=np.random.default_rng(C['probe_seed']).choice(len(imgs),size=C['probe_n'],replace=False)
 generator=torch.Generator(device=DEVICE).manual_seed(C['eval_epsilon_seed'])
 total_sample=total_mu=0.0
 model.eval()
 for first in range(0,len(ids),C['eval_batch']):
  xx=torch.from_numpy(imgs[ids[first:first+C['eval_batch']]]).to(DEVICE,dtype=torch.float32).flatten(1)
  mu,lv=model.encode(xx)
  noise=torch.randn(mu.shape,device=DEVICE,generator=generator)
  zz=mu+noise*torch.exp(0.5*lv)
  total_sample+=F.binary_cross_entropy_with_logits(model.decode_logits(zz),xx,reduction='sum').item()
  total_mu+=F.binary_cross_entropy_with_logits(model.decode_logits(mu),xx,reduction='sum').item()
 return total_sample/len(ids),total_mu/len(ids)

@torch.inference_mode()
def factor_analysis(model):
 """Class conditional means use the actual labels from all 737,280 images."""
 ns,nsc,nr,nx,ny=map(int,sizes[1:])
 nz=C['latent_dim']
 pos=np.zeros((ny*nx,nz)); ps=np.zeros(ny*nx)
 sc=np.zeros((ns*nsc,nz)); ss=np.zeros(ns*nsc)
 rot=np.zeros((ns*nr,nz)); rs=np.zeros(ns*nr)
 kl_sum=np.zeros(nz); var_sum=np.zeros(nz)
 model.eval()
 for first in range(0,len(imgs),C['analysis_batch']):
  end=min(first+C['analysis_batch'],len(imgs))
  xx=torch.from_numpy(imgs[first:end]).to(DEVICE,dtype=torch.float32).flatten(1)
  mu,lv=model.encode(xx)
  m=mu.cpu().numpy().astype(np.float64)
  l=lv.cpu().numpy().astype(np.float64)
  v=np.exp(l)
  c=labels[first:end].astype(np.int32)
  pi=c[:,5]*nx+c[:,4]
  si=c[:,1]*nsc+c[:,2]
  ri=c[:,1]*nr+c[:,3]
  np.add.at(pos,pi,m); ps+=np.bincount(pi,minlength=len(ps))
  np.add.at(sc,si,m); ss+=np.bincount(si,minlength=len(ss))
  np.add.at(rot,ri,m); rs+=np.bincount(ri,minlength=len(rs))
  kl_sum+=(0.5*(m*m+v-1-l)).sum(0)
  var_sum+=v.sum(0)
 assert np.all(ps>0) and np.all(ss>0) and np.all(rs>0)
 pos=(pos/ps[:,None]).reshape(ny,nx,nz)
 sc=(sc/ss[:,None]).reshape(ns,nsc,nz)
 rot=(rot/rs[:,None]).reshape(ns,nr,nz)
 var_mean=var_sum/len(imgs)
 order=np.lexsort((np.arange(nz),var_mean))
 return {'position_yx':pos,'position_x':pos.mean(0),'position_y':pos.mean(1),
         'scale_shape':sc,'rotation_shape':rot,'KL_per_latent':kl_sum/len(imgs),
         'posterior_variance_per_latent':var_mean,'latent_order':order}

@torch.inference_mode()
def anchor_analysis(model):
 """One fixed real dSprites image for all traversals and both β values."""
 model.eval()
 x=torch.from_numpy(imgs[anchor_id:anchor_id+1]).to(DEVICE,dtype=torch.float32).flatten(1)
 mu,lv=model.encode(x)
 levels=torch.tensor(C['traversal_values'],device=DEVICE,dtype=mu.dtype)
 rows=[]
 for latent in range(C['latent_dim']):
  z=mu.repeat(len(levels),1)
  z[:,latent]=levels
  frames=torch.sigmoid(model.decode_logits(z)).reshape(len(levels),C['side'],C['side'])
  rows.append(frames.cpu().numpy().astype(np.float32))
 baseline=torch.sigmoid(model.decode_logits(mu)).reshape(C['side'],C['side'])
 return {'traversals':np.stack(rows), 'traversal_levels':levels.cpu().numpy(),
         'anchor_original':imgs[anchor_id].copy(), 'anchor_decoded_mean':baseline.cpu().numpy(),
         'anchor_mu':mu.cpu().numpy().squeeze(), 'anchor_logvar':lv.cpu().numpy().squeeze(),
         'anchor_factor_classes':labels[anchor_id].copy(),
         'anchor_factor_values':values[anchor_id].copy()}

def analyze_run(beta,seed):
 d=folder(beta,seed)
 p=d/'checkpoint.pt'; mpath=d/'metrics.json'; apath=d/'analysis_metrics.npz'
 if apath.exists() and mpath.exists():
  print('Already analyzed:',d.name); return
 snap=torch.load(p,map_location='cpu',weights_only=False)
 assert snap['step']==C['train_steps'] and snap['dataset_sha']==DATA_SHA
 model=BetaVAE(snap['config']).to(DEVICE)
 model.load_state_dict(snap['model']); model.eval()
 sampled,at_mu=probe_recon(model)
 factors=factor_analysis(model)
 traversals=anchor_analysis(model)
 # analysis_metrics.npz is a numerical metrics artifact generated FROM this trained checkpoint.
 tmp=d/'analysis_metrics.partial.npz'
 np.savez_compressed(tmp,**factors,**traversals)
 os.replace(tmp,apath)
 result={'beta':beta,'seed':seed,'training_steps':snap['step'],
         'reconstruction_nll_sampled_z_per_image':sampled,
         'reconstruction_nll_at_mu_per_image':at_mu,
         'reconstruction_nll_sampled_z_per_pixel':sampled/C['input_dim'],
         'KL_per_latent_full_dataset':factors['KL_per_latent'].tolist(),
         'KL_total_full_dataset':float(factors['KL_per_latent'].sum()),
         'posterior_variance_per_latent':factors['posterior_variance_per_latent'].tolist(),
         'latent_order_smallest_variance_first':factors['latent_order'].tolist(),
         'dataset_sha256':DATA_SHA,'data_download_url':C['download_url'],
         'probe_is_in_sample':True,'probe_size':C['probe_n'],
         'anchor_index':anchor_id,'anchor_factor_classes':labels[anchor_id].tolist(),
         'traversal_values':C['traversal_values'],
         'plot':C['plot'],'shape_names':C['shape_names'],'top_count':C['top_count'],
         'history':snap['history'], 'analysis_file':apath.name, 'checkpoint_file':p.name}
 write_json(result,mpath)
 del model
 torch.cuda.empty_cache()
 print(f'Metrics saved for β={beta:g} seed={seed}: BCE {sampled:.3f}, total KL {result["KL_total_full_dataset"]:.3f}')

## 6. RUN ALL 6 REPLICAS: train, checkpoint, then evaluate

In [ ]:
for beta in C['betas']:
 for seed in C['seeds']:
  print(f'\n=== β={beta:g} seed={seed} ===')
  fit(beta,seed)
  analyze_run(beta,seed)
print('All requested runs complete; no models were filtered.')

## 7. Reconstruction loss and ten per-dimension KL values for every run — reads saved metrics only

In [ ]:
import pandas as pd
records=[]
for meta_path in sorted(ROOT.glob('beta*_seed*/metrics.json')):
 with open(meta_path) as f: m=json.load(f)
 item={'beta':m['beta'],'seed':m['seed'],'steps':m['training_steps'],
       'sampled_BCE_per_image':m['reconstruction_nll_sampled_z_per_image'],
       'mean_z_BCE_per_image':m['reconstruction_nll_at_mu_per_image'],
       'KL_total':m['KL_total_full_dataset']}
 item.update({f'KL_z{j}':float(x) for j,x in enumerate(m['KL_per_latent_full_dataset'])})
 records.append(item)
report=pd.DataFrame(records).sort_values(['beta','seed']).reset_index(drop=True)
assert set(zip(report.beta,report.seed))=={(b,s) for b in C['betas'] for s in C['seeds']}
print('BCE: summed pixel NLL per image on one fixed 16,384-image IN-SAMPLE probe. KL: exact full-dataset Gaussian KL.')
print('Per-dimension KLs are NOT multiplied by β. Training was unsupervised.')
display(report.round(4))
report.to_csv(ROOT/'run_metrics.csv',index=False)
summary=report.groupby('beta')[['sampled_BCE_per_image','mean_z_BCE_per_image','KL_total']].agg(['mean','std'])
print('Mean and standard deviation between random seeds:')
display(summary.round(4))
summary.to_csv(ROOT/'seed_summary.csv')

## 8. Figure 7-style plotting **from saved metrics only** (no numeric constants in plotting code)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def read_run(d):
 with open(d/'metrics.json',encoding='utf-8') as f: m=json.load(f)
 with np.load(d/m['analysis_file'],allow_pickle=False) as arrays:
  a={key:arrays[key].copy() for key in arrays.files}
 return m,a

def figure_seven(d):
 m,a=read_run(d)
 st=m['plot']
 order=list(a['latent_order'])
 selected=order[:m['top_count']]+[next(reversed(order))]
 levels=list(reversed(a['traversal_levels']))
 first_row,scale_row,rotation_row=range(len(st['top_ratios']))
 ratios=st['top_ratios']+[st['image_ratio']]*len(levels)
 fig=plt.figure(figsize=st['main_size'],dpi=st['dpi'])
 layout=GridSpec(len(ratios),len(selected),figure=fig,height_ratios=ratios,
                 hspace=st['hspace'],wspace=st['wspace'])
 for col,latent in enumerate(selected):
  unused=latent==next(reversed(order))
  title=f'z{latent} (unused example)' if unused else f'z{latent}'
  ax=fig.add_subplot(layout[first_row,col])
  ax.imshow(a['position_yx'][...,latent],origin='lower',cmap=st['cmap_position'])
  ax.set_title(title,fontsize=st['title_font']); ax.set_xticks([]); ax.set_yticks([])
  if col==first_row:
   ax.set_ylabel('Position Y',fontsize=st['label_font'])
   ax.set_xlabel('Position X',fontsize=st['label_font'])
  ax=fig.add_subplot(layout[scale_row,col])
  for line,name,color in zip(a['scale_shape'][...,latent],m['shape_names'],st['shape_colors']):
   ax.plot(line,label=name,color=color,alpha=st['alpha'],linewidth=st['line_width'])
  ax.set_xticks(st['scale_xticks']); ax.tick_params(labelsize=st['small_font'])
  if col==first_row:
   ax.set_ylabel('Mean μ, scale',fontsize=st['label_font'])
   ax.legend(fontsize=st['small_font'])
  ax=fig.add_subplot(layout[rotation_row,col])
  for line,name,color in zip(a['rotation_shape'][...,latent],m['shape_names'],st['shape_colors']):
   ax.plot(line,label=name,color=color,alpha=st['alpha'],linewidth=st['line_width'])
  ax.set_xticks(st['rotation_xticks']); ax.tick_params(labelsize=st['small_font'])
  if col==first_row: ax.set_ylabel('Mean μ, rotation',fontsize=st['label_font'])
  for row,(level,image) in enumerate(zip(levels,reversed(a['traversals'][latent])),
                                      start=len(st['top_ratios'])):
   ax=fig.add_subplot(layout[row,col])
   ax.imshow(image,cmap=st['cmap_image'],vmin=st['image_low'],vmax=st['image_high'])
   ax.set_xticks([]); ax.set_yticks([])
   if col==first_row:
    ax.set_ylabel(f'{level:g}',fontsize=st['small_font'],rotation='horizontal')
 fig.suptitle(f'β={m["beta"]:g} | seed={m["seed"]} | ascending posterior variance | anchor={m["anchor_factor_classes"]}',
              fontsize=st['title_font'])
 path=d/'figure7_reproduction.png'
 fig.savefig(path,dpi=st['dpi'],bbox_inches='tight')
 return fig,path

def all_factor_responses(d):
 m,a=read_run(d); st=m['plot']; order=list(a['latent_order'])
 labels=['Position Y×X','X marginal','Y marginal','Scale by shape','Rotation by shape']
 fig,axs=plt.subplots(len(labels),len(order),figsize=st['factor_size'],dpi=st['dpi'])
 rows=dict(zip(labels,axs))
 first_col=next(iter(range(len(order))))
 for col,latent in enumerate(order):
  rows['Position Y×X'][col].imshow(a['position_yx'][...,latent],origin='lower',cmap=st['cmap_position'])
  rows['Position Y×X'][col].set_title(f'z{latent} | KL={m["KL_per_latent_full_dataset"][latent]:g}',
                                      fontsize=st['small_font'])
  rows['X marginal'][col].plot(a['position_x'][:,latent],linewidth=st['line_width'])
  rows['Y marginal'][col].plot(a['position_y'][:,latent],linewidth=st['line_width'])
  for key,signal in [('Scale by shape',a['scale_shape']),('Rotation by shape',a['rotation_shape'])]:
   for line,name,color in zip(signal[...,latent],m['shape_names'],st['shape_colors']):
    rows[key][col].plot(line,label=name,color=color,linewidth=st['line_width'])
  for ax in axs[:,col]: ax.tick_params(labelsize=st['small_font'])
 for key in labels: rows[key][first_col].set_ylabel(key,fontsize=st['label_font'])
 fig.suptitle(f'All latent factor responses | β={m["beta"]:g}, seed={m["seed"]}',fontsize=st['title_font'])
 fig.subplots_adjust(wspace=st['wspace'],hspace=st['hspace'])
 path=d/'all_factor_responses.png'
 fig.savefig(path,dpi=st['dpi'],bbox_inches='tight')
 return fig,path

def all_latent_traversals(d):
 m,a=read_run(d); st=m['plot']; order=list(a['latent_order'])
 levels=list(reversed(a['traversal_levels']))
 fig,axs=plt.subplots(len(levels),len(order),figsize=st['traversal_size'],dpi=st['dpi'])
 first_col=next(iter(range(len(order))))
 for col,latent in enumerate(order):
  next(iter(axs[:,col])).set_title(f'z{latent}',fontsize=st['title_font'])
  for ax,img,level in zip(axs[:,col],reversed(a['traversals'][latent]),levels):
   ax.imshow(img,cmap=st['cmap_image'],vmin=st['image_low'],vmax=st['image_high'])
   ax.set_xticks([]); ax.set_yticks([])
   if col==first_col: ax.set_ylabel(f'{level:g}',rotation='horizontal',fontsize=st['small_font'])
 fig.suptitle(f'All latent traversals | β={m["beta"]:g}, seed={m["seed"]}',fontsize=st['title_font'])
 fig.subplots_adjust(hspace=st['hspace'],wspace=st['wspace'])
 path=d/'all_latent_traversals.png'
 fig.savefig(path,dpi=st['dpi'],bbox_inches='tight')
 return fig,path

## 9. Render Figure 7A / 7B and all supplementary plots from saved artifacts

In [ ]:
# The predetermined representative is the lowest seed for each β, NOT the best looking run.
representative_folders=[folder(beta,min(C['seeds'])) for beta in C['betas']]
for d in representative_folders:
 for fn in [figure_seven,all_factor_responses,all_latent_traversals]:
  fig,path=fn(d)
  display(fig)
  plt.close(fig)
  print('Saved:',path)
# Preserve a Figure 7-style comparison for EVERY other replica too.
for beta in C['betas']:
 for seed in C['seeds']:
  d=folder(beta,seed)
  if d not in representative_folders:
   fig,path=figure_seven(d)
   plt.close(fig)
   print('Additional-seed panel:',path)

## 10. Artifact manifest

In [ ]:
manifest={'source_paper':'Higgins et al. ICLR 2017, Figure 7A/7B',
          'official_data_url':C['download_url'],'dataset_sha256':DATA_SHA,
          'training_steps_are_author_specified':False,
          'figure7_anchor_is_author_specified':False,
          'runs':[str(folder(b,s)) for b in C['betas'] for s in C['seeds']],
          'per_run':['checkpoint.pt','training_history.json','metrics.json',
                     'analysis_metrics.npz','figure7_reproduction.png'],
          'summary_files':['run_metrics.csv','seed_summary.csv']}
write_json(manifest,ROOT/'experiment_manifest.json')
print(json.dumps(manifest,indent=2))